# Model Training Pipeline

Kevin Han, Noah Son, Sophia Fu

Convolutional autoencoder (U-Net style with skip connections) for predicting
finger flexion from ECoG spectrograms.  Matches the full-size architecture from
`example_notebooks/Lightning_BCI-autoencoder.ipynb`.

**Architecture highlights**
- Input: `(batch, channels, 40 freqs, time)` spectrogram windows
- Spatial reduction Conv → Encoder (5 × stride-2 Conv1d) → Decoder (5 × Upsample + Conv1d) with skip connections
- Output: `(batch, 5 fingers, time)` — same temporal resolution as input
- Loss: `0.5 × MSE + 0.5 × (1 − cosine_similarity)` on **4 fingers only** (ring finger excluded)
- Post-processing: Gaussian smoothing (σ = 6) on predicted sequences

In [1]:
import pathlib, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
import scipy.ndimage
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT   = pathlib.Path('..').resolve()
CLEAN_DATA_DIR = PROJECT_ROOT / 'cleaned_data'
PRED_DIR       = PROJECT_ROOT / 'predictions'
CKPT_DIR       = PROJECT_ROOT / 'checkpoints'
PRED_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

DEVICE       = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
N_SUBJECTS   = 3
N_FINGERS    = 5
EVAL_FINGERS = [0, 1, 2, 4]          # skip ring finger (index 3)
LOSS_FINGERS = torch.tensor([0, 1, 2, 4])  # same mask applied during training
FINGER_NAMES = ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']

# ── Training hyperparameters ──────────────────────────────────────────────
SAMPLE_LEN   = 256   # window length in samples at 100 Hz (= 2.56 s, matches example)
TRAIN_STRIDE = 5     # stride between training windows (= 50 ms)
BATCH_SIZE   = 32
MAX_EPOCHS   = 30
LR           = 8.42e-5   # matches example notebook lr
TRAIN_FRAC   = 0.8

# ── Model hyperparameters (matches example notebook) ─────────────────────
# Example: channels=[32,32,64,64,128,128], 5 stages, ~660K params
N_FREQS = 40
MODEL_HP = dict(
    channels     = [32, 32, 64, 64, 128, 128],
    kernel_sizes = [7, 7, 5, 5, 5],
    strides      = [2, 2, 2, 2, 2],
    dilation     = [1, 1, 1, 1, 1],
    n_freqs      = N_FREQS,
    n_channels_out = N_FINGERS,
)

print(f'Device: {DEVICE}')

Device: mps


## Dataset

Sliding-window sampler over the `(channels, freqs, T)` spectrograms.
Training windows use `TRAIN_STRIDE=5` to reduce epoch length; validation
windows use stride=1 for complete coverage.

In [2]:
class EcogDataset(Dataset):
    """
    Sliding-window dataset.
    specs : (channels, freqs, T)
    ff    : (5, T)
    """
    def __init__(self, specs: np.ndarray, ff: np.ndarray,
                 sample_len: int, stride: int = 1):
        self.specs = torch.from_numpy(specs.astype('float32'))
        self.ff    = torch.from_numpy(ff.astype('float32'))
        T = specs.shape[2]
        self.sample_len = sample_len
        self.starts = list(range(0, T - sample_len, stride))

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s = self.starts[idx]
        e = s + self.sample_len
        return self.specs[..., s:e], self.ff[..., s:e]


def make_loaders(specs, ff,
                 sample_len=SAMPLE_LEN, batch_size=BATCH_SIZE,
                 train_frac=TRAIN_FRAC, train_stride=TRAIN_STRIDE):
    """Chronological 80/20 split -> (train_loader, val_loader)."""
    T   = specs.shape[2]
    cut = int(T * train_frac)
    train_ds = EcogDataset(specs[..., :cut], ff[..., :cut],
                           sample_len=sample_len, stride=train_stride)
    val_ds   = EcogDataset(specs[..., cut:],  ff[..., cut:],
                           sample_len=sample_len, stride=1)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=0)
    print(f'  Train windows: {len(train_ds):>6}  '
          f'Val windows: {len(val_ds):>6}  (T={T}, split at t={cut})')
    return train_loader, val_loader

## Model Architecture

Same U-Net style as the example notebook, scaled down:

```
Input  (B, C, F, T)  ─► reshape ─► (B, C×F, T)
                         spatial_reduce ConvBlock
                         Encoder: 3 × ConvBlock (stride-2 MaxPool)
                         Decoder: 3 × UpConvBlock (Upsample ×2) + skip-cat
                         Conv1×1 ─► (B, 5, T)
```

The `n_electrodes` parameter is set per-subject because the three subjects
have different channel counts (62, 48, 64).

In [3]:
class ConvBlock(nn.Module):
    """Conv1d → LayerNorm → GELU → Dropout → MaxPool."""
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, dilation=1, p_drop=0.1):
        super().__init__()
        self.conv   = nn.Conv1d(in_channels, out_channels, kernel_size,
                                bias=False, padding='same')
        self.norm   = nn.LayerNorm(out_channels)
        self.act    = nn.GELU()
        self.drop   = nn.Dropout(p=p_drop)
        self.pool   = nn.MaxPool1d(kernel_size=stride, stride=stride)

    def forward(self, x):
        x = self.conv(x)
        x = self.norm(x.transpose(-2, -1)).transpose(-2, -1)
        x = self.act(x)
        x = self.drop(x)
        x = self.pool(x)
        return x


class UpConvBlock(nn.Module):
    """ConvBlock followed by linear Upsample."""
    def __init__(self, scale, **conv_kwargs):
        super().__init__()
        self.conv   = ConvBlock(**conv_kwargs)
        self.up     = nn.Upsample(scale_factor=scale, mode='linear',
                                  align_corners=False)

    def forward(self, x):
        return self.up(self.conv(x))


class AutoEncoder1D(nn.Module):
    """
    U-Net style 1-D convolutional autoencoder with skip connections.
    Input : (batch, n_electrodes, n_freqs, time)
    Output: (batch, n_channels_out, time)
    """
    def __init__(self, n_electrodes, n_freqs=40, n_channels_out=5,
                 channels=None, kernel_sizes=None,
                 strides=None, dilation=None):
        super().__init__()
        if channels is None:
            channels = [8, 16, 32, 32]
        depth = len(channels) - 1

        self.n_inp = n_freqs * n_electrodes
        self.depth = depth

        # Spatial reduction: collapse electrode×freq dimension
        self.spatial_reduce = ConvBlock(self.n_inp, channels[0], kernel_size=3)

        # Encoder
        self.encoder = nn.ModuleList([
            ConvBlock(channels[i], channels[i+1],
                      kernel_sizes[i], stride=strides[i], dilation=dilation[i])
            for i in range(depth)
        ])

        # Decoder — deepest block takes channels[-1] directly;
        #           all others take 2× (skip connection doubles the channels)
        self.decoder = nn.ModuleList([
            UpConvBlock(
                scale=strides[i],
                in_channels=channels[i+1] if i == depth - 1 else channels[i+1] * 2,
                out_channels=channels[i],
                kernel_size=kernel_sizes[i],
            )
            for i in range(depth - 1, -1, -1)
        ])

        # Final 1×1 conv: (channels[0] * 2) -> n_channels_out
        self.head = nn.Conv1d(channels[0] * 2, n_channels_out,
                              kernel_size=1, padding='same')

    def forward(self, x):
        B, C, F, T = x.shape
        x = x.reshape(B, -1, T)          # (B, C×F, T)
        x = self.spatial_reduce(x)        # (B, channels[0], T)

        skips = []
        for block in self.encoder:
            skips.append(x)
            x = block(x)

        for i, block in enumerate(self.decoder):
            x = block(x)
            # Trim skip to match upsampled length (guards against odd-length inputs)
            skip = skips[-(i + 1)]
            t    = min(x.shape[-1], skip.shape[-1])
            x    = torch.cat([x[..., :t], skip[..., :t]], dim=1)

        return self.head(x)               # (B, n_channels_out, T)

## Loss and Inference Utilities

In [4]:
def cosine_sim(y_hat, y):
    """Mean cosine similarity along the time axis."""
    return torch.mean(nn.CosineSimilarity(dim=-1)(y_hat, y))


def combined_loss(y_hat, y):
    """
    0.5 * MSE + 0.5 * (1 - cosine_sim), computed only on the 4 evaluated
    fingers (thumb, index, middle, pinky — ring finger index 3 is excluded).

    y_hat, y : (batch, 5, time)
    """
    idx = LOSS_FINGERS.to(y_hat.device)
    y_hat_eval = y_hat[:, idx, :]   # (batch, 4, time)
    y_eval     = y[:, idx, :]       # (batch, 4, time)
    mse  = F.mse_loss(y_hat_eval, y_eval)
    corr = cosine_sim(y_hat_eval, y_eval)
    return 0.5 * mse + 0.5 * (1.0 - corr)


# stride_multiple = 2^n_stages so skip-connection sizes always match
_STRIDE_MULTIPLE = 2 ** len(MODEL_HP['strides'])

def predict_sequence(model, specs, device=DEVICE):
    """
    Run the model on a full-length spectrogram in a single forward pass.
    Trims T to a multiple of _STRIDE_MULTIPLE so encoder/decoder sizes align.

    specs : (channels, freqs, T)
    returns: (T, 5) numpy array
    """
    model.eval()
    T = specs.shape[2]
    T_trim = (T // _STRIDE_MULTIPLE) * _STRIDE_MULTIPLE
    with torch.no_grad():
        x = (torch.from_numpy(specs[..., :T_trim].astype('float32'))
             .unsqueeze(0).to(device))          # (1, C, F, T_trim)
        out = model(x)[0].cpu().numpy()         # (5, T_trim)
    # Pad trimmed tail with last prediction value
    if T_trim < T:
        out = np.concatenate([out,
                              np.repeat(out[:, -1:], T - T_trim, axis=1)], axis=1)
    return out.T   # (T, 5)


def evaluate_correlations(preds, ff, smooth_sigma=6):
    """
    Gaussian-smooth predictions then compute per-finger Pearson r.
    preds : (T, 5)   ff : (5, T)
    Returns list of 5 correlations.
    """
    smooth = scipy.ndimage.gaussian_filter1d(preds.T, sigma=smooth_sigma).T
    return [pearsonr(ff[f], smooth[:, f])[0] for f in range(N_FINGERS)]

## Training Loop

`train_subject` trains one subject from scratch, checkpoints the best
validation loss, and returns the loaded best model.

In [5]:
def train_subject(subj_idx, specs_train, ff_train, n_electrodes,
                  max_epochs=MAX_EPOCHS, lr=LR):
    print(f"\n{'='*55}")
    print(f'  Subject {subj_idx + 1}  |  '
          f'electrodes={n_electrodes}  '
          f'n_input_feats={n_electrodes * N_FREQS}')
    print(f"{'='*55}")

    train_loader, val_loader = make_loaders(specs_train, ff_train)

    hp    = {**MODEL_HP, 'n_electrodes': n_electrodes}
    model = AutoEncoder1D(**hp).to(DEVICE)
    n_p   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable parameters: {n_p:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=max_epochs)

    best_val   = float('inf')
    ckpt_path  = CKPT_DIR / f'subj{subj_idx + 1}_best.pt'

    for epoch in range(1, max_epochs + 1):
        # ── Train ──────────────────────────────────────────────────────
        model.train()
        t_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = combined_loss(model(x), y)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
        t_loss /= len(train_loader)

        # ── Validate ───────────────────────────────────────────────────
        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                v_loss += combined_loss(model(x), y).item()
        v_loss /= len(val_loader)
        scheduler.step()

        if v_loss < best_val:
            best_val = v_loss
            torch.save(model.state_dict(), ckpt_path)

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{max_epochs}  '
                  f'train={t_loss:.4f}  val={v_loss:.4f}  '
                  f'best={best_val:.4f}')

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    print(f'  Loaded best checkpoint  (val_loss={best_val:.4f})')
    return model

## Load Data and Train All Subjects

In [6]:
all_data   = {}
all_models = {}

for subj in range(N_SUBJECTS):
    subj_dir = CLEAN_DATA_DIR / f'subj{subj + 1}'
    all_data[subj] = {
        'specs_train': np.load(subj_dir / 'specs_train.npy'),
        'ff_train':    np.load(subj_dir / 'ff_train.npy'),
        'specs_lead':  np.load(subj_dir / 'specs_lead.npy'),
    }

for subj in range(N_SUBJECTS):
    all_models[subj] = train_subject(
        subj_idx     = subj,
        specs_train  = all_data[subj]['specs_train'],
        ff_train     = all_data[subj]['ff_train'],
        n_electrodes = all_data[subj]['specs_train'].shape[0],
    )


  Subject 1  |  electrodes=62  n_input_feats=2480
  Train windows:   4746  Val windows:   5740  (T=29980, split at t=23984)
  Trainable parameters: 652,613
  Epoch   1/30  train=0.0988  val=0.0420  best=0.0420
  Epoch   5/30  train=0.0243  val=0.0387  best=0.0387
  Epoch  10/30  train=0.0113  val=0.0350  best=0.0350
  Epoch  15/30  train=0.0084  val=0.0368  best=0.0324
  Epoch  20/30  train=0.0071  val=0.0335  best=0.0324
  Epoch  25/30  train=0.0065  val=0.0344  best=0.0324
  Epoch  30/30  train=0.0064  val=0.0347  best=0.0324
  Loaded best checkpoint  (val_loss=0.0324)

  Subject 2  |  electrodes=48  n_input_feats=1920
  Train windows:   4746  Val windows:   5740  (T=29980, split at t=23984)
  Trainable parameters: 598,853
  Epoch   1/30  train=0.0947  val=0.0489  best=0.0489
  Epoch   5/30  train=0.0225  val=0.0518  best=0.0448
  Epoch  10/30  train=0.0112  val=0.0466  best=0.0448
  Epoch  15/30  train=0.0083  val=0.0489  best=0.0448
  Epoch  20/30  train=0.0070  val=0.0475  best=0

## Evaluate on Validation Set

Predictions are made on the full validation sequence, Gaussian-smoothed
(σ = 6 samples, matching the example), then scored with Pearson *r*.
Ring finger (index 3) is excluded from the summary metric.

In [7]:
results = []

for subj in range(N_SUBJECTS):
    model = all_models[subj]
    specs = all_data[subj]['specs_train']
    ff    = all_data[subj]['ff_train']
    T     = specs.shape[2]
    cut   = int(T * TRAIN_FRAC)

    preds  = predict_sequence(model, specs[..., cut:])  # (T_val, 5)
    ff_val = ff[..., cut:]                              # (5, T_val)
    corrs  = evaluate_correlations(preds, ff_val)

    print(f'\nSubject {subj + 1}:')
    for f_idx in range(N_FINGERS):
        skip = '  [SKIP]' if f_idx == 3 else ''
        print(f'  {FINGER_NAMES[f_idx]:8s}: r = {corrs[f_idx]:+.4f}{skip}')
        results.append({'subject': subj + 1, 'finger': FINGER_NAMES[f_idx],
                        'f_idx': f_idx, 'corr': corrs[f_idx]})

df = pd.DataFrame(results)
eval_corrs = df[df['f_idx'].isin(EVAL_FINGERS)]['corr']
print(f"\n{'='*45}")
print(f'  Mean correlation (12 fingers): {eval_corrs.mean():.4f}')
print(f"{'='*45}")


Subject 1:
  Thumb   : r = +0.6297
  Index   : r = +0.8280
  Middle  : r = +0.4538
  Ring    : r = +0.3267  [SKIP]
  Pinky   : r = +0.0891

Subject 2:
  Thumb   : r = +0.4320
  Index   : r = +0.2964
  Middle  : r = +0.1594
  Ring    : r = -0.3162  [SKIP]
  Pinky   : r = +0.1268

Subject 3:
  Thumb   : r = +0.7463
  Index   : r = +0.4804
  Middle  : r = +0.5236
  Ring    : r = -0.1717  [SKIP]
  Pinky   : r = +0.6321

  Mean correlation (12 fingers): 0.4498


## Save Leaderboard Predictions

In [8]:
for subj in range(N_SUBJECTS):
    preds    = predict_sequence(all_models[subj],
                                all_data[subj]['specs_lead'])
    out_path = PRED_DIR / f'subj{subj + 1}_leaderboard_pred.npy'
    np.save(out_path, preds)
    print(f'Subject {subj + 1}: saved {preds.shape} -> {out_path.name}')

Subject 1: saved (14750, 5) -> subj1_leaderboard_pred.npy
Subject 2: saved (14750, 5) -> subj2_leaderboard_pred.npy
Subject 3: saved (14750, 5) -> subj3_leaderboard_pred.npy
